In [1]:
import os

In [2]:
pwd

'c:\\Users\\User\\Documents\\chest-cancer-classification\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'c:\\Users\\User\\Documents\\chest-cancer-classification'

In [6]:
from dataclasses import dataclass
from pathlib import Path

In [7]:
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir : Path
    base_model_path : Path
    updated_base_model_path : Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int



In [8]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier import logger


In [9]:
class ConfigurationManager:
    def __init__(self, config_path = CONFIG_FILE_PATH, params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])
    
    def get_prepare_base_model_config(self):
        prepare_base_model_config = self.config.prepare_base_model

        create_directories([prepare_base_model_config.root_dir])


        return PrepareBaseModelConfig(
            prepare_base_model_config.root_dir,
            prepare_base_model_config.base_model_path, 
            prepare_base_model_config.updated_base_model_path,  
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )


In [10]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

[2026-05-30 14:31:30,198: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
]


In [17]:
class PrepareBaseModel:
    def __init__(self, prepare_base_model_config):
        self.config = prepare_base_model_config
    
    def get_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.model.save(self.config.base_model_path)
    
    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            model.trainable = False
        elif (freeze_till is not None) and freeze_till > 0:
            for layer in model.layers[:freeze_till]:
                layer.trainable = False
        
        flattern_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units = classes,
            activation = "softmax"
        )(flattern_in)

        full_model = tf.keras.models.Model(
            inputs = model.input,
            outputs = prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        self.full_model.save(self.config.updated_base_model_path)

In [18]:
try:
    configuration_manager = ConfigurationManager()
    base_model_config = configuration_manager.get_prepare_base_model_config()
    base_model = PrepareBaseModel(base_model_config)
    base_model.get_model()
    base_model.update_base_model()
except Exception as e:
    raise e

[2026-05-30 17:35:17,447: INFO : common : line 35 : yaml file: config\config.yaml loaded successfully]
[2026-05-30 17:35:17,454: INFO : common : line 35 : yaml file: params.yaml loaded successfully]
[2026-05-30 17:35:17,458: INFO : common : line 56 : created directory: artifacts]
[2026-05-30 17:35:17,462: INFO : common : line 56 : created directory: artifacts/prepare_base_model]
[2026-05-30 17:35:17,752: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\backend.py:1398: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
]
[2026-05-30 17:35:18,144: WARNING : module_wrapper : line 149 : From c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\layers\pooling\max_pooling2d.py:161: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.
]
58889256/58889256 [=======================

c:\Users\User\Documents\chest-cancer-classification\.venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     